### The Multiple Knapsack Problem

The multiple knapsack problem involves a set of $N$ items and a set of $M$ knapsacks, where $M \le N$. The objective is to select $M$ disjoint subsets of items to maximize the total profit, ensuring that each subset is assigned to a different knapsack and does not exceed its capacity. When $M=1$, this reduces to the standard knapsack problem.

**Parameters:**
* $w_j$: weight of item $j$
* $v_j$: value of item $j$
* $C_i$: capacity of knapsack $i$

#### Mathematical Model

**Decision Variables:**
The binary variable $x_{ij}$ equals $1$ if item $j$ is assigned to knapsack $i$, and $0$ otherwise:
$$x_{ij} \in \{0, 1\} \quad \text{for } i=1,\dots,M \text{ and } j=1,\dots,N$$

**Objective Function:**
Maximize the total profit of the assigned items:
$$\text{maximize } z = \sum_{i=1}^{M}\sum_{j=1}^{N}v_{j}x_{ij}$$

**Constraints:**
1. Capacity Constraint: The total weight of items assigned to knapsack $i$ cannot exceed its capacity $C_i$:
$$\sum_{j=1}^{N}w_{j}x_{ij} \le C_{i} \quad i=1, 2, \dots, M$$

2. Assignment Constraint: Each item $j$ can be assigned to at most one knapsack:
$$\sum_{i=1}^{M}x_{ij} \le 1 \quad j=1, 2, \dots, N$$

In [1]:
import numpy as np
import scipy.sparse as sp

import cplex as cp

In [2]:
def mixed_integer_linear_programming(direction, A, senses, b, c, l, u, types, names):
    # create an empty optimization problem
    prob = cp.Cplex()

    # add decision variables to the problem including their coefficients in objective and ranges
    prob.variables.add(obj = c.tolist(), lb = l.tolist(), ub = u.tolist(), types = types.tolist(), names = names.tolist())

    # define problem type
    if direction == "maximize":
        prob.objective.set_sense(prob.objective.sense.maximize)
    else:
        prob.objective.set_sense(prob.objective.sense.minimize)

    # add constraints to the problem including their directions and right-hand side values
    prob.linear_constraints.add(senses = senses.tolist(), rhs = b.tolist())

    # add coefficients for each constraint
    row_indices, col_indices = A.nonzero()
    prob.linear_constraints.set_coefficients(zip(row_indices.tolist(), col_indices.tolist(), A.data.tolist()))

    print(prob.write_as_string())
    # solve the problem
    prob.solve()
    
    # check the solution status
    print(prob.solution.get_status())
    print(prob.solution.status[prob.solution.get_status()])

    # get the solution
    x_star = prob.solution.get_values()
    obj_star = prob.solution.get_objective_value()

    return(x_star, obj_star)


In [3]:
def multiple_knapsack_problem(weights_file, values_file, capacities_file):
    # your implementation starts below
    weights=np.loadtxt(weights_file)
    values=np.loadtxt(values_file)
    capacities=np.loadtxt(capacities_file)
    N=len(values)
    M=len(capacities)

    c=np.tile(values,M)
    senses=np.repeat("L",M+N)
    l=np.repeat(0,M*N)
    u=np.repeat(1,M*N)
    b=np.concatenate((capacities,np.repeat(1,N)))
    types=np.repeat("B",M*N)
    names=[]
    for i in range (N*M):
        a="x"+str(i)
        names.append(a)
    names=np.array(names)

    aij=np.concatenate((np.tile(weights,M),np.repeat(1,M*N)))
    row=np.concatenate((np.repeat(range(M),N),np.repeat(range(M,N+M),M)))
    col=np.concatenate((range(M*N),np.array(range(M*N)).reshape(M,N).T.flatten()))
    A=sp.csr_matrix((aij,(row,col)),shape=(M+N,M*N))
    x_star, obj_star=mixed_integer_linear_programming("maximize", A, senses, b, c, l, u, types, names)
    X_star=np.array(x_star).reshape(M,N)
    # your implementation ends above
    return(X_star)

In [4]:
X_star = multiple_knapsack_problem("weights.txt", "values.txt", "capacities.txt")
print(X_star)

Default row names c1, c2 ... being created.


\ENCODING=ISO-8859-1
\Problem name: 

Maximize
 obj1: 50 x0 + 40 x1 + 30 x2 + 20 x3 + 10 x4 + 50 x5 + 40 x6 + 30 x7 + 20 x8
       + 10 x9
Subject To
 c1: 10 x0 + 20 x1 + 30 x2 + 40 x3 + 50 x4 <= 40
 c2: 10 x5 + 20 x6 + 30 x7 + 40 x8 + 50 x9 <= 50
 c3: x0 + x5 <= 1
 c4: x1 + x6 <= 1
 c5: x2 + x7 <= 1
 c6: x3 + x8 <= 1
 c7: x4 + x9 <= 1
Bounds
 0 <= x0 <= 1
 0 <= x1 <= 1
 0 <= x2 <= 1
 0 <= x3 <= 1
 0 <= x4 <= 1
 0 <= x5 <= 1
 0 <= x6 <= 1
 0 <= x7 <= 1
 0 <= x8 <= 1
 0 <= x9 <= 1
Binaries
 x0  x1  x2  x3  x4  x5  x6  x7  x8  x9 
End

Version identifier: 22.1.1.0 | 2022-11-28 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 0.000000 after 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 1 rows and 1 columns.
MIP Presolve modified 3 coefficients.
Reduced MIP has 6 rows, 9 columns, and 17 nonzeros.
Reduced MIP has 9 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.02 ticks)
Probing time = 0.00 sec. (